# Semenov et al. (2025), Figures 1 and 2, from dandisets 001032 to 001035

Claims. The `data: elsewhere` claims of `claims/001271/10.1016-j.xcrp.2025.102660.yaml` that
the claims file assigns, from their titles, to DANDI:001032 to 001035 ("Characterization of
Neuron Action Potential Triggering Using Sine Wave Electrostimulation at Varying Frequencies",
parts 1 to 4). Each is pinned at its only published version: 001032 0.240527.1432, 001033
0.240527.1432, 001034 0.240527.1431, 001035 0.240527.1432 (the paper cites no version).

The paper: Semenov I, Kim V, Silkuniene G, Pakhomov AG (2025), *Cell Reports Physical Science* 6,
102660, doi:10.1016/j.xcrp.2025.102660; the claims file was written from the NIH author
manuscript (PMC12320893) with its STAR Methods and supplement.

| Claim | Here |
|---|---|
| fig1-pure-sine-excites-all | tested |
| fig1-20khz-needs-twice-the-field | tested |
| fig2d-modulated-thresholds-higher | tested |
| fig2-modulated-locks-to-beats | tested |
| fig2c-pure-more-aps | tested |
| fig3b-low-conductivity-halves-threshold | blocked: no 250-ms sessions in the low-conductivity solution (checked below) |
| figs1-di8-same-as-fluovolt | tested on DANDI:001272 (0.250805.1358, the most recent published version), at the end |

The AP readings O1 to O4 of the claims file's header are run in every combination, with the
whole 16 x 16 frame as a variant.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import sinewave as sw

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 40)
SETS = ["001032", "001033", "001034", "001035"]
print({d: sw.OTHER_DANDISETS[d] for d in SETS})

## Data check

In [ ]:
t0 = time.time()
parts, trace_list = [], []
for d in SETS:
    meta = sw.dandiset_metadata(d, sw.OTHER_DANDISETS[d])
    print(d, meta["name"], "|", meta["assetsSummary"]["numberOfFiles"], "files |",
          "related:", [r.get("identifier") for r in meta.get("relatedResource", [])])
    assets = sw.list_assets(d, sw.OTHER_DANDISETS[d])
    S_, T_ = sw.load_all(assets, parser=sw.parse_series_description)
    S_["dandiset"] = d
    S_["trace_index"] = S_["trace_index"] + len(trace_list)
    trace_list += T_
    parts.append(S_)
ALL = pd.concat(parts, ignore_index=True)
ALL["n_frames"] = ALL["shape"].apply(lambda s: s[0])
print(f"streamed and reduced {len(ALL)} files in {time.time() - t0:.0f} s")

In [ ]:
print("files per dandiset:", ALL.groupby("dandiset").size().to_dict())
print("frames per file:", ALL.groupby(["dandiset", "n_frames"]).size().to_dict())
print("rates:", ALL["rate"].value_counts().to_dict(), "| units:", ALL["unit"].value_counts().to_dict())
print("solution named in the descriptions:", ALL["solution"].value_counts().to_dict(),
      "| any description naming NaGluconate or low conductance:",
      int(ALL["description"].str.contains("Gluconate|low-conduct", regex=True).sum()))
print("V/cm per V stated:", ALL["vcm_per_volt_file"].value_counts().to_dict())
print("stimulus kinds:", ALL.groupby(["kind", "khz"]).size().to_dict())
dup = ALL.groupby("sha1")["dandiset"].apply(lambda v: tuple(sorted(v)))
print("data arrays found in more than one file:", int((dup.apply(len) > 1).sum()),
      "| dandiset combinations:", dup[dup.apply(len) > 1].value_counts().to_dict())
a = ALL[ALL["dandiset"] == "001032"].set_index("sha1")
b = ALL[ALL["dandiset"] == "001034"].set_index("sha1")
common = a.index.intersection(b.index)
print(f"001034 files whose data equal a 001032 file: {len(common)} of {len(b)}; same description after the "
      f"subject id: {int((a.loc[common, 'description'].str.split('. ', n=1, regex=False).str[1] == b.loc[common, 'description'].str.split('. ', n=1, regex=False).str[1]).sum())}; "
      f"same start time: {int((a.loc[common, 'start'] == b.loc[common, 'start']).sum())}")
print("example subject ids, 001032 against 001034:", list(zip(a.loc[common[:3], "subject"], b.loc[common[:3], "subject"])))
S = ALL[(ALL["dandiset"] != "001034") & ~ALL["sham"]].copy()
print("\nused: 001032, 001033 and 001035 (001034 duplicates 001032), sham sessions left out:", len(S), "sessions,",
      S["subject"].nunique(), "subjects")
fm = np.array(ALL["frame_means"].tolist())
print(f"frame 0 / frames 1 to n-2: {np.min(fm[:, 0] / fm[:, 1]):.2f} to {np.max(fm[:, 0] / fm[:, 1]):.2f}")
tab = S.groupby(["subject"]).apply(lambda g: pd.Series({f"{k} {f:g} kHz": int(((g['kind'] == k) & (g['khz'] == f)).sum())
                                                        for k, f in [("sine", 2), ("spindle", 2), ("sine", 20), ("spindle", 20)]}))
tab["other"] = S.groupby("subject").apply(lambda g: int((~g["khz"].isin([2, 20])).sum()))
print(tab.to_string())
print("subjects with each kind:", (tab > 0).sum().to_dict())
print("voltage range per kind:", S.groupby(["kind", "khz"])["volt"].agg(["min", "max"]).loc[[("sine", 2), ("spindle", 2), ("sine", 20), ("spindle", 20)]].to_dict("index"))

Subject identity: the mean pre-stimulus frames of sessions within a subject against those of
other subjects. Stimulus timing: AP-like peaks in the whole-frame trace by time from the start of
the recording; the Methods give the onset at 50 ms and a 250-ms stimulus.

In [ ]:
mp = np.vstack([m.ravel() for m in S["mean_pre"]])
cm = np.corrcoef(mp)
sa = S["subject"].astype(str).to_numpy(dtype=object)
same = sa[:, None] == sa[None, :]
iu = np.triu_indices(len(S), 1)
print(f"mean-frame correlation within subjects: median {np.median(cm[iu][same[iu]]):.3f}, min {cm[iu][same[iu]].min():.3f}; "
      f"between: median {np.median(cm[iu][~same[iu]]):.3f}, max {cm[iu][~same[iu]].max():.3f}")
subj_list = sorted(S["subject"].unique())
idx = {k: np.where(sa == k)[0] for k in subj_list}
low_within = [(k, round(float(cm[np.ix_(v, v)][np.triu_indices(len(v), 1)].min()), 3)) for k, v in idx.items() if len(v) > 1]
print("subjects whose least similar pair of sessions correlates below 0.8:", [x for x in low_within if x[1] < 0.8])
close = []
for i, k1 in enumerate(subj_list):
    for k2 in subj_list[i + 1:]:
        v = cm[np.ix_(idx[k1], idx[k2])].max()
        if v > 0.95:
            close.append((k1, k2, round(float(v), 3)))
print("pairs of subjects with some sessions correlating above 0.95:", close)
for n in sorted(S["n_frames"].unique()):
    lat = []
    sub = S[S["n_frames"] == n]
    for _, r in sub.iterrows():
        x, _ = sw.dff(trace_list[r["trace_index"]]["frame"], r["rate"], "all")
        lat += list(sw.detect_aps(np.nan_to_num(x), r["rate"], 5.0, 0, len(x))["frame"] / r["rate"] * 1000)
    h, e = np.histogram(lat, bins=np.arange(0, n / 3.0864 + 25, 25))
    print(f"{n}-frame sessions ({len(sub)}, recording {n / 3.0864:.0f} ms): peaks per 25-ms bin from 0 ms:", h.tolist())

## Readings and thresholds

Per session, APs are detected during the stimulus (50 to 300 ms, or to the end of a shorter
recording) under every combination of O1 to O4 (sinewave.measure). The threshold of a neuron for
a stimulus (kind and carrier) and an AP count N (1 or 5) is the lowest voltage at which a session
has at least N APs during the stimulus (Figure 2D legend: "the minimum electric field strength
that elicits 1 or 5 action potentials"); a voltage tested more than once counts when any session
reaches N (a standard choice). Field strength: the files state 36 V/cm per V of a peak-to-peak
voltage; dandiset 001271's voltages match the fields of Figure S2 at 18 V/cm per V (see that
notebook), so both are shown, 18 V/cm per V first.

In [ ]:
S["phase"], S["group"] = "before", "series"
M = sw.all_combinations(S, trace_list)
M = M[M["x"].apply(lambda v: np.isfinite(v).any())].copy()
KEYS = ["roi", "f0", "ap", "sdr"]
CLAIMS_ROI = ("otsu", "otsu_eroded")
print("sessions per ROI reading:", M[M["sdr"] == "5sd"].groupby(["roi", "f0", "ap"]).size().unstack().to_dict())
STIMS = [("sine", 2.0), ("spindle", 2.0), ("sine", 20.0), ("spindle", 20.0)]


def thresholds(m, n_ap):
    out = []
    for (subj, kind, khz), g in m.groupby(["subject", "kind", "khz"]):
        if (kind, khz) not in STIMS:
            continue
        hit = g.loc[g["n_ap"] >= n_ap, "volt"]
        below = g.loc[g["n_ap"] < n_ap, "volt"]
        out.append(dict(subject=subj, kind=kind, khz=khz, n_ap=n_ap, thr_v=hit.min() if len(hit) else np.nan,
                        bracketed=bool(len(hit) and (below < hit.min()).any()), tested=len(g)))
    return pd.DataFrame(out)


THR = {}
for key, m in M.groupby(KEYS):
    if key[3] != "5sd":
        continue
    THR[key[:3]] = pd.concat([thresholds(m, 1), thresholds(m, 5)], ignore_index=True)
t = THR[("otsu", "all", "5pct")]
print("thresholds (V) under otsu/all/5pct: neurons with a threshold, of neurons tested")
print(t.groupby(["kind", "khz", "n_ap"]).agg(neurons=("subject", "size"), with_thr=("thr_v", lambda v: int(v.notna().sum())),
                                              bracketed=("bracketed", "sum"), mean_v=("thr_v", "mean")).round(2).to_string())

## fig1-pure-sine-excites-all

> Unmodulated 250-ms sine waves at 2 and 20 kHz excited every neuron tested, with APs 100 to 200
> ms after onset near threshold, firing rate rising to a ceiling of 60 to 90 Hz, and sustained
> depolarization at the highest fields.

Reproduced when, under every reading, at least 90% of neurons are excited at each frequency, the
median over neurons of the highest maximum firing rate lies within 54 to 99 Hz and the median
first-AP latency at threshold within 90 to 220 ms.

In [ ]:
rows = []
for key, m in M.groupby(KEYS):
    for khz in (2.0, 20.0):
        g = m[(m["kind"] == "sine") & (m["khz"] == khz)]
        per = []
        for subj, h in g.groupby("subject"):
            exc = (h["n_ap"] > 0).any()
            thr = h.loc[h["n_ap"] > 0, "volt"].min() if exc else np.nan
            lat = h.loc[(h["volt"] == thr) & (h["n_ap"] > 0), "first_ap_ms"].min() if exc else np.nan
            top = h[h["volt"] == h["volt"].max()]
            per.append(dict(subject=subj, excited=exc, ceiling=h["max_rate_hz"].max(), latency=lat,
                            sd_top=bool(top["sustained"].any())))
        p = pd.DataFrame(per)
        frac = 100 * p["excited"].mean()
        med_ceiling = p["ceiling"].median()
        med_lat = p["latency"].median()
        ok = frac >= 90 and 54 <= med_ceiling <= 99 and 90 <= med_lat <= 220
        rows.append(dict(zip(KEYS, key), khz=khz, neurons=len(p), excited_pct=round(frac, 1),
                         median_ceiling_hz=round(med_ceiling, 1), ceiling_range=f"{p['ceiling'].min():.0f}-{p['ceiling'].max():.0f}",
                         neurons_ceiling_in_54_99=int(p["ceiling"].between(54, 99).sum()),
                         median_latency_ms=round(med_lat, 1), sd_at_top=int(p["sd_top"].sum()), ok=ok))
r1 = pd.DataFrame(rows)
print(r1.to_string(index=False))
c = r1[r1["roi"].isin(CLAIMS_ROI)].groupby(KEYS)["ok"].all()
print(f"\nreadings (claims-file ROIs) where both frequencies pass: {int(c.sum())} of {len(c)}")
print("parts failing, claims-file ROIs: excited < 90%:", int((r1[r1['roi'].isin(CLAIMS_ROI)]['excited_pct'] < 90).sum()),
      "| ceiling outside 54-99:", int((~r1[r1['roi'].isin(CLAIMS_ROI)]['median_ceiling_hz'].between(54, 99)).sum()),
      "| latency outside 90-220:", int((~r1[r1['roi'].isin(CLAIMS_ROI)]['median_latency_ms'].between(90, 220)).sum()),
      "of", int(r1["roi"].isin(CLAIMS_ROI).sum()), "reading x frequency rows")

**Result: not reproduced.** No combination of the claims file's readings meets the criterion at
both frequencies (0 of 16). With the O1(a) ROI, 96.4% (5% height) or 89.3% (5-SD height) of 28
neurons are excited at 2 kHz and 88.2% or 76.5% of 17 at 20 kHz, so excitation is common but
not universal at 20 kHz; the median highest firing rate is 22.2 to 36.7 Hz, below the 54 to
99 Hz the criterion asks, and few neurons reach it (1 to 5 per frequency); the median latency of
the first AP at threshold is 53.5 to 70.0 ms at 2 kHz and 67.4 to 123.8 ms at 20 kHz, against 90
to 220 ms. Under O1(b) (14 and 10 neurons) the ceilings are 21.7 to 53.9 Hz. Sustained
depolarization (O4) at the highest field is found in at most one neuron.

## fig1-20khz-needs-twice-the-field

> All excitation thresholds at 20 kHz were about 2-fold higher than at 2 kHz.

Ratio of the mean 20-kHz to the mean 2-kHz threshold for each stimulus (unmodulated, modulated)
and AP count (1, 5), within 1.8 to 2.2, and 20 against 2 kHz at p < 0.0001 (two-tailed unpaired t
test): reading (a) one test per stimulus and AP count, (b) one test pooling the four. The ratio
does not depend on the V/cm conversion.

In [ ]:
rows = []
for key, t in THR.items():
    pooled2, pooled20, all_ok = [], [], True
    for kind in ("sine", "spindle"):
        for n in (1, 5):
            a2 = t[(t["kind"] == kind) & (t["khz"] == 2.0) & (t["n_ap"] == n)]["thr_v"].dropna()
            a20 = t[(t["kind"] == kind) & (t["khz"] == 20.0) & (t["n_ap"] == n)]["thr_v"].dropna()
            pooled2 += list(a2)
            pooled20 += list(a20)
            ratio = a20.mean() / a2.mean() if len(a2) and len(a20) else np.nan
            p = stats.ttest_ind(a20, a2).pvalue if len(a2) > 1 and len(a20) > 1 else np.nan
            rows.append(dict(roi=key[0], f0=key[1], ap=key[2], stim=kind, n_ap=n, n2=len(a2), n20=len(a20),
                             mean2_v=round(a2.mean(), 2), mean20_v=round(a20.mean(), 2), ratio=round(ratio, 2), p=p,
                             ratio_ok=bool(1.8 <= ratio <= 2.2), p_ok=bool(p < 1e-4)))
    pp = stats.ttest_ind(pooled20, pooled2).pvalue if len(pooled20) > 1 else np.nan
    rows.append(dict(roi=key[0], f0=key[1], ap=key[2], stim="pooled", n_ap=0, n2=len(pooled2), n20=len(pooled20),
                     mean2_v=round(np.mean(pooled2), 2), mean20_v=round(np.mean(pooled20), 2),
                     ratio=round(np.mean(pooled20) / np.mean(pooled2), 2), p=pp, ratio_ok=np.nan, p_ok=bool(pp < 1e-4)))
r2 = pd.DataFrame(rows)
print(r2.to_string(index=False))
cl = r2[r2["roi"].isin(CLAIMS_ROI) & (r2["stim"] != "pooled")]
print(f"\nclaims-file ROIs: ratios within 1.8-2.2 in {int(cl['ratio_ok'].sum())} of {len(cl)} stimulus x AP-count rows; "
      f"p < 0.0001 in {int(cl['p_ok'].sum())} of {len(cl)}; pooled p < 0.0001 in "
      f"{int(r2[r2['roi'].isin(CLAIMS_ROI) & (r2['stim'] == 'pooled')]['p_ok'].sum())} of {int((r2['roi'].isin(CLAIMS_ROI) & (r2['stim'] == 'pooled')).sum())}")
print(f"ratio range, claims-file ROIs, unmodulated: {cl[cl['stim'] == 'sine']['ratio'].min()} to {cl[cl['stim'] == 'sine']['ratio'].max()}; "
      f"modulated: {cl[cl['stim'] == 'spindle']['ratio'].min()} to {cl[cl['stim'] == 'spindle']['ratio'].max()}")

**Result: not reproduced.** Thresholds at 20 kHz are higher than at 2 kHz in every reading
(pooled p < 0.0001 in 7 of 8 readings), but the ratio is within 1.8 to 2.2 in only 6 of 32
stimulus and AP-count rows, and in no reading for all four. The unmodulated 1-AP ratio is 2.15 to
2.19 with the 5% height (within), and 2.56 (O1 a) or 3.51 to 3.53 (O1 b) with the 5-SD height; the unmodulated 5-AP
ratios are 2.06 to 2.84 and the modulated ones 2.24 to 2.96, with 1 to 3 neurons at 20 kHz. The
ratios are of the stated generator voltages; the files say the actual amplitude is smaller by a
frequency-dependent calibration factor that is not given, so the field ratio may differ.

## fig2d-modulated-thresholds-higher

> Thresholds for sine waves modulated at 20 Hz were significantly higher than for pure sine waves
> at the same carrier, for 5 APs at 2 kHz and for 1 and 5 APs at 20 kHz, so modulation did not
> lower excitation thresholds; the 2-kHz unmodulated 1-AP threshold was 57.7 V/cm.

Two-tailed unpaired t tests, modulated against unmodulated. Reproduced when the three are higher
with p < 0.05, the 2-kHz 1-AP threshold is not lower for modulated waves at p < 0.05, and the
mean 2-kHz unmodulated 1-AP threshold is within 57.7 +/- 3.6 V/cm, under every AP reading.

In [ ]:
rows = []
for key, t in THR.items():
    rec = dict(roi=key[0], f0=key[1], ap=key[2])
    ok = True
    for khz, n, need in [(2.0, 1, "not lower"), (2.0, 5, "higher"), (20.0, 1, "higher"), (20.0, 5, "higher")]:
        u = t[(t["kind"] == "sine") & (t["khz"] == khz) & (t["n_ap"] == n)]["thr_v"].dropna()
        mo = t[(t["kind"] == "spindle") & (t["khz"] == khz) & (t["n_ap"] == n)]["thr_v"].dropna()
        res = stats.ttest_ind(mo, u) if len(u) > 1 and len(mo) > 1 else None
        p = res.pvalue if res is not None else np.nan
        d = mo.mean() - u.mean()
        good = (d > 0 and p < 0.05) if need == "higher" else not (d < 0 and p < 0.05)
        ok &= bool(good)
        rec[f"{khz:g}k{n}"] = f"{u.mean():.2f}/{mo.mean():.2f} V n={len(u)}/{len(mo)} p={p:.2g}"
        rec[f"{khz:g}k{n}_ok"] = bool(good)
    u1 = t[(t["kind"] == "sine") & (t["khz"] == 2.0) & (t["n_ap"] == 1)]["thr_v"].dropna()
    u5 = t[(t["kind"] == "sine") & (t["khz"] == 2.0) & (t["n_ap"] == 5)]["thr_v"].dropna()
    for k in (18, 36):
        rec[f"thr1_vcm_{k}"] = round(k * u1.mean(), 1)
        rec[f"thr1_sem_{k}"] = round(k * u1.std(ddof=1) / np.sqrt(len(u1)), 1)
        rec[f"thr5_vcm_{k}"] = round(k * u5.mean(), 1)
    rec["thr_ok_18"] = bool(abs(18 * u1.mean() - 57.7) <= 3.6)
    rec["thr_ok_36"] = bool(abs(36 * u1.mean() - 57.7) <= 3.6)
    rec["tests_ok"] = ok
    rows.append(rec)
r3 = pd.DataFrame(rows)
print(r3.to_string(index=False))
cl = r3[r3["roi"].isin(CLAIMS_ROI)]
print(f"\nclaims-file ROIs: the four test conditions hold in {int(cl['tests_ok'].sum())} of {len(cl)} readings; "
      f"mean 1-AP threshold within 57.7 +/- 3.6 V/cm in {int(cl['thr_ok_18'].sum())} (18 V/cm per V) and "
      f"{int(cl['thr_ok_36'].sum())} (36 V/cm per V) of {len(cl)}")

**Result: not reproduced.** Under no reading do the three comparisons the paper reports as
significant all hold. Modulated thresholds are higher on average in every comparison that can be
made, but the 2-kHz 5-AP difference is not significant (p = 0.082 to 0.28) and at 20 kHz only 1
to 3 neurons have modulated thresholds (1-AP p = 0.013 with the 5% height under O1(a), 0.80 with
the 5-SD height; the 5-AP test cannot be run). The 2-kHz 1-AP threshold is not lower for
modulated waves in any reading. The mean 2-kHz unmodulated 1-AP threshold is 74.1 to 79.0 V/cm
under O1(a) at 18 V/cm per V (61.2 to 67.6 under O1(b)), against 57.7 +/- 3.6; at the files'
36 V/cm per V it is twice that.

## fig2-modulated-locks-to-beats

> With 20-Hz modulation, firing above threshold locked to the beats, one AP near the peak of each
> of the five beats and at most two per beat (ten in all), and modulated sine waves triggered no
> sustained firing or sustained depolarization.

Modulated sessions only. Reproduced when no session has more than 10 APs, in sessions with 5 APs
or more most APs fall within 12.5 to 37.5 ms of each 50-ms beat, and no session shows sustained
firing or sustained depolarization (O4). As the claims file now has it (amended after a reader
answered the question in `questions/001271.yaml`), a modulated session shows sustained firing when
its firing is not bound to the beats: more than two APs in one beat, or, in a session with 5 APs
or more, at least as many APs outside 12.5 to 37.5 ms of the beats as inside. The header's O4
definition of sustained firing (at least three APs with one in the last 50 ms), which the first
run used, is no longer the criterion; it is still counted below for comparison.

In [ ]:
def beat_flags(r):
    """Per session: share of APs within 12.5-37.5 ms of a beat, more than two APs in one beat, at least as
    many APs outside that window as inside (sessions with 5 APs or more), and the header's O4 sustained firing."""
    tms = (r["ap_frames"] - r["onset"]) / r["rate"] * 1000
    ph = np.mod(tms, 50)
    near = (ph >= 12.5) & (ph <= 37.5)
    b = (tms // 50).astype(int)
    b = b[(b >= 0) & (b < 5)]
    over2 = bool(len(b) and np.bincount(b, minlength=5).max() > 2)
    off = bool(r["n_ap"] >= 5 and (~near).sum() >= near.sum())
    o4 = bool(r["n_ap"] >= 3 and (tms >= 200).any())
    return np.mean(near) if len(tms) else np.nan, over2, off, o4


rows = []
for key, m in M.groupby(KEYS):
    g = m[m["kind"] == "spindle"]
    f = pd.DataFrame([beat_flags(r) for _, r in g.iterrows()], columns=["near", "over2", "off", "o4"], index=g.index)
    five = f[g["n_ap"] >= 5]
    over10 = int((g["n_ap"] > 10).sum())
    sus_fire = int((f["over2"] | f["off"]).sum())
    sus_dep = int(g["sustained"].sum())
    ok = over10 == 0 and bool((five["near"] > 0.5).all()) and sus_fire == 0 and sus_dep == 0
    rows.append(dict(zip(KEYS, key), sessions=len(g), max_aps=int(g["n_ap"].max()), over10=over10, sessions_5plus=len(five),
                     share_near_peak_median=round(float(five["near"].median()), 2) if len(five) else np.nan,
                     sessions_majority_near_peak=int((five["near"] > 0.5).sum()),
                     beat_over2=int(f["over2"].sum()), outside_ge_inside=int(f["off"].sum()), sustained_firing=sus_fire,
                     sustained_dep=sus_dep, o4_sustained_firing=int(f["o4"].sum()), ok=ok))
r4 = pd.DataFrame(rows)
print(r4.to_string(index=False))
cl = r4[r4["roi"].isin(CLAIMS_ROI)]
print(f"\nclaims-file ROIs: passing readings {int(cl['ok'].sum())} of {len(cl)}; "
      f"under O1(a): {int(cl[cl['roi'] == 'otsu']['ok'].sum())} of {int((cl['roi'] == 'otsu').sum())}")
m = M[(M["roi"] == "otsu") & (M["f0"] == "all") & (M["ap"] == "5pct") & (M["sdr"] == "5sd") & (M["kind"] == "spindle")]
ph = np.concatenate([np.mod((r["ap_frames"] - r["onset"]) / r["rate"] * 1000, 50) for _, r in m.iterrows()])
beat = np.concatenate([((r["ap_frames"] - r["onset"]) / r["rate"] * 1000 // 50) for _, r in m.iterrows()])
print(f"\notsu/all/5pct, modulated sessions: {len(ph)} APs; share within 12.5-37.5 ms of a beat {np.mean((ph >= 12.5) & (ph <= 37.5)):.2f} "
      f"(0.50 by chance); APs per beat 1-5: {np.bincount(beat.astype(int), minlength=5)[:5].tolist()}")
per_beat = []
for _, r in m.iterrows():
    b = ((r["ap_frames"] - r["onset"]) / r["rate"] * 1000 // 50).astype(int)
    per_beat += list(np.bincount(b[(b >= 0) & (b < 5)], minlength=5))
print("APs per beat over all modulated sessions and beats:", pd.Series(per_beat).value_counts().sort_index().to_dict())
h, e = np.histogram(ph, bins=np.arange(0, 51, 5))
print("AP phase within the 50-ms beat, 5-ms bins:", h.tolist())
un = M[(M["roi"] == "otsu") & (M["f0"] == "all") & (M["ap"] == "5pct") & (M["sdr"] == "5sd") & (M["kind"] == "sine") & M["khz"].isin([2.0, 20.0])]
phu = np.concatenate([np.mod((r["ap_frames"] - r["onset"]) / r["rate"] * 1000, 50) for _, r in un.iterrows()])
print(f"unmodulated 2 and 20 kHz, for comparison: {len(phu)} APs, share within 12.5-37.5 ms {np.mean((phu >= 12.5) & (phu <= 37.5)):.2f}; "
      f"sessions over 10 APs {int((un['n_ap'] > 10).sum())} of {len(un)}")
fu = pd.DataFrame([beat_flags(r) for _, r in un.iterrows()], columns=["near", "over2", "off", "o4"])
ub = (fu["over2"] | fu["off"]).to_numpy()
print(f"unmodulated sessions that the amended rule calls sustained firing: {int(sum(ub))} of {len(un)} "
      f"({int(ub[(un['n_ap'] >= 3).to_numpy()].sum())} of {int((un['n_ap'] >= 3).sum())} with 3 APs or more)")
flag = m[[beat_flags(r)[1] for _, r in m.iterrows()]]
for _, r in flag.iterrows():
    print(f"\nsession with more than two APs in a beat (otsu/all/5pct): subject {r['subject']}, {r['khz']:g} kHz, {r['volt']:g} V, "
          f"{r['n_ap']} APs, pre-onset SD {r['pre_sd']:.2f}% dF")
    for apr in ("5pct", "5sd"):
        q = M[(M["roi"] == "otsu") & (M["f0"] == "all") & (M["ap"] == apr) & (M["sdr"] == "5sd") & (M["path"] == r["path"])].iloc[0]
        print(f"  {apr}: level {q['ap_level']:.2f}% dF; AP times (ms from onset)",
              np.round((q["ap_frames"] - q["onset"]) / q["rate"] * 1000, 1).tolist(),
              "| peaks (% dF)", np.round(q["ap_peaks"], 1).tolist(), "| widths (ms)", np.round(q["ap_widths"], 1).tolist())
    hv = M[(M["roi"] == "otsu") & (M["f0"] == "all") & (M["ap"] == "5pct") & (M["sdr"] == "5sd") & (M["subject"] == r["subject"])]
    print("  median AP peak of this neuron over its sessions (% dF):", round(float(np.median(np.concatenate(list(hv["ap_peaks"])))), 1))

**Result: reproduced (resting on an assumption).** Under the amended criterion, 12 of the 16
readings pass: all four O1(a) readings with the 5-SD height and all eight O1(b) readings. Firing
locks to the beats: with O1(a) and the 5% height, 92% of 212 APs in modulated sessions fall within
12.5 to 37.5 ms of a beat (49% for unmodulated waves), no modulated session has more than 8 APs
(26 of 370 unmodulated sessions have more than 10), every session with 5 APs or more has most of
its APs near the beat peaks (14 of 14), no session has at least as many APs outside that window as
inside, and none shows sustained depolarization. The four O1(a) readings with the 5% height fail
through one session, subject 9_2 at 2 kHz and 3.4 V, with three APs in the fifth beat (206.4, 217.4
and 236.2 ms). That session's pre-onset SD is 9.8% dF, about twice the 5% level; the three peaks are
11.1 to 22.2% dF, and none passes the session's 5-SD level (49.0% dF). The 5-SD readings are adopted
for this claim, as the 5% level lies inside that session's noise (an assumption). The first run's
test of sustained firing, the header's O4 (at least three APs with one in the last 50 ms), would
flag 15 to 32 modulated sessions under O1(a); the reader's answer replaced it, since one AP per beat
meets it. The amended rule calls 88 of the 197 unmodulated sessions with 3 APs or more sustained
firing, so it does separate the two kinds of wave.

## fig2c-pure-more-aps

> In the same neuron at the same field strength, pure sine waves triggered more APs than
> modulated ones, and the threshold was typically lower for unmodulated signals.

Pairs are a neuron's modulated and unmodulated sessions at the same carrier and voltage.
Reproduced when the unmodulated count is larger in more pairs than the reverse, and the
unmodulated threshold lower in more neurons than the reverse, under every reading. The threshold
is taken for 1 AP and for 5 APs (the claim does not say which; both are run).

In [ ]:
rows = []
for key, m in M.groupby(KEYS):
    if key[3] != "5sd":
        continue
    g = m[m["khz"].isin([2.0, 20.0])]
    pr = []
    for (subj, khz, v), h in g.groupby(["subject", "khz", "volt"]):
        if set(h["kind"]) >= {"sine", "spindle"}:
            pr.append(dict(subject=subj, khz=khz, volt=v, unmod=h.loc[h["kind"] == "sine", "n_ap"].max(),
                           mod=h.loc[h["kind"] == "spindle", "n_ap"].max()))
    pr = pd.DataFrame(pr)
    t = THR[key[:3]]
    rec = dict(roi=key[0], f0=key[1], ap=key[2], pairs=len(pr), neurons=pr["subject"].nunique(),
               unmod_more=int((pr["unmod"] > pr["mod"]).sum()), mod_more=int((pr["mod"] > pr["unmod"]).sum()),
               p_paired=stats.ttest_rel(pr["unmod"], pr["mod"]).pvalue if len(pr) > 1 else np.nan)
    for n in (1, 5):
        w = (t[t["n_ap"] == n].pivot_table(index=["subject", "khz"], columns="kind", values="thr_v")
             .reindex(columns=["sine", "spindle"]).dropna())
        rec[f"thr{n}_pairs"] = len(w)
        rec[f"thr{n}_unmod_lower"] = int((w["sine"] < w["spindle"]).sum())
        rec[f"thr{n}_mod_lower"] = int((w["spindle"] < w["sine"]).sum())
    rec["ok"] = (rec["unmod_more"] > rec["mod_more"]) and all(rec[f"thr{n}_unmod_lower"] > rec[f"thr{n}_mod_lower"] for n in (1, 5))
    rows.append(rec)
r5 = pd.DataFrame(rows)
print(r5.to_string(index=False))
print(f"\nclaims-file ROIs: passing readings {int(r5[r5['roi'].isin(CLAIMS_ROI)]['ok'].sum())} of {int(r5['roi'].isin(CLAIMS_ROI).sum())}")

**Result: reproduced (resting on an assumption).** With the O1(a) ROI, under all four F0 and
height readings, the unmodulated wave gives more APs than the modulated one at the same carrier
and voltage in 32 to 43 of 52 pairs against 2 to 4 (paired t p = 2.3e-10 to 7.8e-07), and the
unmodulated threshold is lower in 10 to 13 of 12 to 15 neurons for 1 AP (against 0 to 2) and in
2 to 6 of 4 to 9 for 5 APs (against 0). The O1(b) ROI leaves 5 neurons and at most one 5-AP
threshold pair, and fails there; the O1(a) readings are adopted as the better supported ones.

## fig3b-low-conductivity-halves-threshold

The claim needs 1-AP thresholds for 250-ms unmodulated sine waves in the NaCl and the
low-conductivity NaGlu solution at each frequency. The descriptions of 001032 to 001035 name the
standard NaCl solution only (data check above); 001030, the only NaGlu dandiset of the paper, holds
2-s ramped 2-kHz waves in four cells and no NaCl sessions.

In [ ]:
print("sessions per solution named, 001032 to 001035:", ALL["solution"].value_counts().to_dict())
print("sessions naming NaGluconate:", int(ALL["description"].str.contains("Gluconate").sum()))

**Result: blocked (metadata_insufficient).** None of the 803 sessions of 001032 to 001035 is in the
low-conductivity solution, and 001030 holds only 2-s ramped waves at 2 kHz in that solution, so
no NaCl and NaGlu thresholds for 250-ms waves at 2 to 20 kHz can be formed. The paper does not say
which dandiset holds Figures 3A and 3B, and none of those read here holds them.

## figs1-di8-same-as-fluovolt (DANDI:001272)

> With Di-8-ANEPPS in place of FluoVolt, in eight neurons, the effects of 2-kHz sine waves were
> not different, so the excitation by unmodulated sine waves is not a sensitization by FluoVolt.

The criterion follows the claims file of DANDI:001272: `figs1-eight-neurons` (8 neurons,
tolerance 0, each recording counted once) and `figs1-effects-same-as-fluovolt` (the response
pattern of three claims of that file, and the 1-AP threshold of the Di-8-ANEPPS neurons against
that of the FluoVolt neurons stimulated by 250-ms unmodulated 2-kHz waves in NaCl, two-sided
unpaired t test at p < 0.05). Di-8-ANEPPS traces are inverted (that file's F1). The neuron count
is tested in full; of the second claim, the threshold test is run here and the three pattern
claims are not. Thresholds are compared in the stated voltages, assuming the two dandisets state
them on the same scale.

In [ ]:
A72 = sw.list_assets("001272", sw.OTHER_DANDISETS["001272"])
S72, T72 = sw.load_all(A72)
S72 = S72.sort_values(["subject", "start"])
print("001272:", len(S72), "files; subjects:", S72.groupby("subject").size().to_dict())
print("descriptions name Di-8-ANEPPS:", int(S72["description"].str.contains("Di-8-ANEPPS").sum()), "| frequency (kHz):",
      S72["khz"].value_counts().to_dict(), "| sham:", int(S72["sham"].sum()), "| distinct data arrays:", S72["sha1"].nunique())
mp72 = np.vstack([m.ravel() for m in S72["mean_pre"]])
c72 = np.corrcoef(mp72)
s72 = S72["subject"].astype(str).to_numpy(dtype=object)
w72 = c72[s72[:, None] == s72[None, :]].min()
b72 = c72[s72[:, None] != s72[None, :]].max()
print(f"mean-frame correlation: within subjects min {w72:.3f}, between subjects max {b72:.3f}")
print("voltages per neuron:", S72.groupby("subject")["volt"].apply(lambda v: sorted(set(v))).to_dict())
n_di8 = S72["subject"].nunique()
print(f"Di-8-ANEPPS neurons: {n_di8} (expected 8, tolerance 0)")
S72["phase"], S72["group"] = "before", "di8"
rows = []
for roi in ("otsu", "otsu_eroded"):
    for f0r in ("all", "last20"):
        for apr in ("5pct", "5sd"):
            m72 = sw.measure(S72[~S72["sham"]], T72, roi, f0r, apr, "5sd", invert=True)
            m72 = m72[m72["x"].apply(lambda v: np.isfinite(v).any())]
            d8 = m72[m72["n_ap"] >= 1].groupby("subject")["volt"].min().astype(float)
            fv = THR[(roi, f0r, apr)]
            fv = fv[(fv["kind"] == "sine") & (fv["khz"] == 2.0) & (fv["n_ap"] == 1)]["thr_v"].dropna()
            d8v = d8.dropna()
            fv = fv.astype(float)
            p = stats.ttest_ind(d8v, fv).pvalue if len(d8v) > 1 and len(fv) > 1 else np.nan
            rows.append(dict(roi=roi, f0=f0r, ap=apr, di8_neurons_with_ap=len(d8v), di8_thr_v=np.round(d8v.values, 2).tolist(),
                             di8_mean=round(d8v.mean(), 2), fluovolt_n=len(fv), fluovolt_mean=round(fv.mean(), 2),
                             diff=round(d8v.mean() - fv.mean(), 2), p=p,
                             di8_aps=int(m72["n_ap"].sum()), di8_sessions=len(m72)))
r6 = pd.DataFrame(rows)
print(r6.to_string(index=False))

**Result: not reproduced.** DANDI:001272 holds 3 Di-8-ANEPPS neurons, three distinct cells by
their images, where the paper reports eight and the criterion allows no other count. On these
three, under the O1(a) readings, the mean 1-AP threshold is 3.73 to 4.23 V against 4.11 to 4.39 V
for 25 to 27 FluoVolt neurons (difference -0.38 to 0.10 V, p = 0.72 to 0.93), so the test finds no
difference; the O1(b) ROI is empty in every 001272 session. The pattern claims of the 001272
claims file are not run here.

## Summary

| Claim | Outcome |
|---|---|
| fig1-pure-sine-excites-all | not reproduced |
| fig1-20khz-needs-twice-the-field | not reproduced |
| fig2d-modulated-thresholds-higher | not reproduced |
| fig2-modulated-locks-to-beats | reproduced (resting on an assumption; criterion amended after a reader's answer) |
| fig2c-pure-more-aps | reproduced (resting on an assumption) |
| fig3b-low-conductivity-halves-threshold | blocked: metadata insufficient (no dandiset found to hold it) |
| figs1-di8-same-as-fluovolt | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

Differences from the paper's method. Thresholds come from the sessions as deposited (the
lowest voltage with at least 1 or 5 APs), with APs found by the claims file's rule on Otsu ROIs of
16 x 16 crops, where the paper drew the soma and read APs by eye. Voltages are the stated
generator values, converted at 18 V/cm per V (and 36 shown), without the frequency-dependent
calibration the files mention but do not give.

About the dandisets. All 178 data arrays of 001034 equal those of 001032, under other subject
identifiers; they are counted once. In 001032 some subjects hold sessions of more than one cell
(13_1's 20-kHz sessions show the cell of 14_1). No session of 001032 to 001035 is in the
low-conductivity solution. 001272 holds 3 Di-8-ANEPPS neurons, where the paper reports 8. 001032 to
001035 do not name the paper in their metadata; 001272 does.